# LeadHarvest AI - Week 1 + Week 2 (Scraping + Cleaning)
Developer: Ahmad
This notebook includes:
- Google Maps, LinkedIn, Yellow Pages, Website Scrapers
- Cleaning & Deduplication Pipeline
- FastAPI App with Endpoints
- Automatic Cleaning + Deduplication inside API responses
- Extra Logging (total_raw, total_cleaned, duplicates_removed)


In [1]:

!pip install fastapi uvicorn scrapy beautifulsoup4 playwright selenium requests pandas nest_asyncio pyngrok duckduckgo-search
!playwright install


Playwright Host validation warning: 
╔══════════════════════════════════════════════════════╗
║ Host system is missing dependencies to run browsers. ║
║ Missing libraries:                                   ║
║     libwoff2dec.so.1.0.2                             ║
║     libgstgl-1.0.so.0                                ║
║     libgstcodecparsers-1.0.so.0                      ║
║     libavif.so.13                                    ║
║     libharfbuzz-icu.so.0                             ║
║     libenchant-2.so.2                                ║
║     libsecret-1.so.0                                 ║
║     libhyphen.so.0                                   ║
║     libmanette-0.2.so.0                              ║
╚══════════════════════════════════════════════════════╝
    at validateDependenciesLinux (/usr/local/lib/python3.12/dist-packages/playwright/driver/package/lib/server/registry/dependencies.js:269:9)
    at process.processTicksAndRejections (node:internal/process/task_queues:105

In [2]:
import os

SERPAPI_KEY = "a11986b6c3012cc964a3eed47daadc9770e568943fd3c1854e8f79551a709c3c"
NGROK_AUTHTOKEN = "31enlFXtoUfKMtrZZHfbNZKG8O0_GCeoScZqRCMqNG9qGzFB"

# also set as environment variables
os.environ["SERPAPI_KEY"] = SERPAPI_KEY
os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTHTOKEN

print("SERPAPI_KEY set:", bool(SERPAPI_KEY))
print("NGROK_AUTHTOKEN set:", bool(NGROK_AUTHTOKEN))


SERPAPI_KEY set: True
NGROK_AUTHTOKEN set: True


In [3]:

%%writefile scraping_googlemaps.py
import requests

def scrape_google_maps(industry: str, location: str):
    # Use SerpAPI if key available
    import os
    SERPAPI_KEY = os.environ.get("SERPAPI_KEY", "a11986b6c3012cc964a3eed47daadc9770e568943fd3c1854e8f79551a709c3c")
    if SERPAPI_KEY:
        url = "https://serpapi.com/search.json"
        params = {"engine":"google_maps","q":f"{industry} in {location}","api_key":SERPAPI_KEY}
        r = requests.get(url, params=params)
        data = r.json().get("local_results", [])
        results = []
        for d in data:
            results.append({
                "name": d.get("title"),
                "address": d.get("address"),
                "phone": d.get("phone"),
                "website": d.get("website"),
                "rating": d.get("rating"),
                "reviews": d.get("reviews"),
                "gps": d.get("gps_coordinates"),
                "email": None
            })
        return results
    else:
        # fallback: OpenStreetMap Nominatim
        url = f"https://nominatim.openstreetmap.org/search"
        params = {"q": f"{industry}, {location}", "format": "json", "limit": 5}
        r = requests.get(url, params=params, headers={"User-Agent":"Mozilla/5.0"})
        data = r.json()
        return [{"name": d.get("display_name"), "location": location, "phone": None, "website": None, "email": None} for d in data]


Overwriting scraping_googlemaps.py


In [4]:

%%writefile scraping_linkedin.py
import requests
from bs4 import BeautifulSoup

def scrape_linkedin(industry: str, location: str, max_results: int = 5):
    query = f"site:linkedin.com/company {industry} {location}"
    url = "https://lite.duckduckgo.com/50x.html"  # Lite version works better
    resp = requests.post(url, data={"q": query}, headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(resp.text, "html.parser")

    results = []
    for res in soup.select("a.result-link")[:max_results]:
        results.append({
            "name": res.get_text(strip=True),
            "url": res.get("href"),
            "location": location,
            "industry": industry
        })

    return results


Overwriting scraping_linkedin.py


In [5]:
%%writefile scraping_yellowpages.py
import requests

def scrape_yellow_pages(industry: str, location: str, max_results: int = 10):
    query = f"{industry} {location}"
    url = f"https://api.opencorporates.com/v0.4/companies/search?q={query}"
    resp = requests.get(url, timeout=15)

    if resp.status_code != 200:
        print("⚠️ API request failed:", resp.status_code)
        return []

    data = resp.json()
    companies = data.get("results", {}).get("companies", [])

    results = []
    for c in companies[:max_results]:
        company = c["company"]
        results.append({
            "name": company.get("name"),
            "location": company.get("jurisdiction_code"),
            "industry": industry,
            "website": company.get("website"),
            "email": None
        })

    return results


Overwriting scraping_yellowpages.py


In [6]:

%%writefile scraping_websites.py
import requests
from bs4 import BeautifulSoup
import re

def scrape_website(url: str):
    try:
        headers = {"User-Agent": "Mozilla/5.0 (LeadHarvestAI/1.0)"}
        resp = requests.get(url, headers=headers, timeout=15)
        if resp.status_code != 200:
            print(f"⚠️ Failed to fetch {url}: {resp.status_code}")
            return []

        soup = BeautifulSoup(resp.text, "html.parser")

        title = soup.title.string.strip() if soup.title else url

        meta = soup.find("meta", attrs={"name": "description"}) \
            or soup.find("meta", attrs={"property": "og:description"})
        description = meta["content"].strip() if meta and "content" in meta.attrs else soup.get_text()[:150]

        text = soup.get_text()
        emails = re.findall(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", text)
        email = emails[0] if emails else None

        phones = re.findall(r"\+?\d[\d\-\(\) ]{7,}\d", text)
        phone = phones[0] if phones else None

        return [{
            "name": title,
            "website": url,
            "email": email,
            "phone": phone,
            "description": description
        }]
    except Exception as e:
        print(f"❌ Error scraping {url}: {e}")
        return []


Overwriting scraping_websites.py


In [7]:

%%writefile cleaning_pipeline.py
import re
import pandas as pd

def normalize_email(email):
    if not email: return None
    return email.strip().lower()

def normalize_phone(phone):
    if not phone: return None
    digits = re.sub(r'\D', '', phone)
    if digits.startswith("00"):
        digits = digits[2:]
    if not digits.startswith("+"):
        digits = "+" + digits
    return digits

def normalize_website(website):
    if not website: return None
    site = re.sub(r"https?://", "", website)
    site = site.strip().lower().rstrip("/")
    return site

def clean_and_deduplicate(raw_results):
    df = pd.DataFrame(raw_results)
    total_raw = len(df)
    if "email" in df.columns:
        df["email"] = df["email"].apply(normalize_email)
    if "phone" in df.columns:
        df["phone"] = df["phone"].apply(normalize_phone)
    if "website" in df.columns:
        df["website"] = df["website"].apply(normalize_website)
    if "name" in df.columns:
        df["name"] = df["name"].fillna("").str.strip().str.title()
    subset_cols = [c for c in ["email","website","name"] if c in df.columns]
    df = df.drop_duplicates(subset=subset_cols, keep="first")
    total_cleaned = len(df)
    duplicates_removed = total_raw - total_cleaned
    return df, {"total_raw": total_raw, "total_cleaned": total_cleaned, "duplicates_removed": duplicates_removed}


Overwriting cleaning_pipeline.py


In [8]:

%%writefile main.py
from fastapi import FastAPI
from scraping_googlemaps import scrape_google_maps
from scraping_linkedin import scrape_linkedin
from scraping_yellowpages import scrape_yellow_pages
from scraping_websites import scrape_website
from cleaning_pipeline import clean_and_deduplicate

app = FastAPI(title="LeadHarvest AI - Week1+2")

@app.get("/")
def root():
    return {"message": "LeadHarvest AI Scraper API with Cleaning & Deduplication"}

@app.get("/scrape/googlemaps")
def google_maps(industry: str, location: str):
    raw = scrape_google_maps(industry, location)
    cleaned, stats = clean_and_deduplicate(raw)
    return {"source": "Google Maps", "stats": stats, "results": cleaned.to_dict(orient="records")}

@app.get("/scrape/linkedin")
def linkedin(industry: str, location: str):
    raw = scrape_linkedin(industry, location)
    cleaned, stats = clean_and_deduplicate(raw)
    return {"source": "LinkedIn", "stats": stats, "results": cleaned.to_dict(orient="records")}

@app.get("/scrape/yellowpages")
def yellowpages(industry: str, location: str):
    raw = scrape_yellow_pages(industry, location)
    cleaned, stats = clean_and_deduplicate(raw)
    return {"source": "Yellow Pages", "stats": stats, "results": cleaned.to_dict(orient="records")}

@app.get("/scrape/website")
def website(url: str):
    raw = scrape_website(url)
    cleaned, stats = clean_and_deduplicate(raw)
    return {"source": "Website", "stats": stats, "results": cleaned.to_dict(orient="records")}





Overwriting main.py


In [ ]:
# kill ngrock
!pkill -f ngrok || echo "No ngrok process found"

import nest_asyncio, uvicorn
from pyngrok import ngrok

# Authenticate ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Start tunnel
public_url = ngrok.connect(8000)
print("🚀 Public API:", public_url)
print("📌 Swagger Docs:", public_url, "/docs")

nest_asyncio.apply()
uvicorn.run("main:app", host="0.0.0.0", port=8000)


^C
🚀 Public API: NgrokTunnel: "https://61e2bc1ca560.ngrok-free.app" -> "http://localhost:8000"
📌 Swagger Docs: NgrokTunnel: "https://61e2bc1ca560.ngrok-free.app" -> "http://localhost:8000" /docs


INFO:     Started server process [28850]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2400:adc7:1135:900:f166:eb62:f54e:45ca:0 - "GET / HTTP/1.1" 200 OK
INFO:     2400:adc7:1135:900:f166:eb62:f54e:45ca:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     2400:adc7:1135:900:f166:eb62:f54e:45ca:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2400:adc7:1135:900:f166:eb62:f54e:45ca:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     2400:adc7:1135:900:f166:eb62:f54e:45ca:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2400:adc7:1135:900:f166:eb62:f54e:45ca:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     124.29.253.184:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     154.208.50.35:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     154.208.50.35:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     154.208.50.35:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     154.208.50.35:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     66.249.93.97:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     66.249.83.33:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     66.249.83.33:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     66.249.83.45:0 -